# Run QPT for DakisSX on real IQM hardware

Adapted from `QPT_SX_old.ipynb`. Loads a calibration produced by `SX_calibration_exp.ipynb`
(a `*_calib.json` + `*_samples.npz` pair in `Results/Calibration/`), runs full
quantum-process-tomography circuits against it and against the native gate (fixed at
its current calibration, for a stable comparison point), reconstructs both Choi
matrices, and saves the result in the same format `view_QPT_results.ipynb` already
knows how to read.

**This notebook submits real jobs to IQM hardware and consumes real shot budget** (14
circuits x `n_shots`, batched into one job). Requires `IQM_TOKEN` set as an environment
variable (see `SX_calibration_exp.ipynb` for details) and the same hardware-SDK packages.

The Choi-reconstruction math here was verified against a real saved QPT result already
in this repo (`Results/QPT/qpt_sx_QB7_garnet_20260718_184944.json`) before being used
in this notebook: recomputing from its raw `results_dakis` reproduces the saved
`process_fidelity` and Choi matrix exactly (`|C_recomputed - C_saved| = 0.0`).


In [ ]:
import os, json, glob
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from datetime import datetime

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Choi, process_fidelity
from qiskit.circuit.library import SXGate

from iqm.qiskit_iqm import IQMProvider
from iqm.pulla.pulla import Pulla
from iqm.pulla.utils_qiskit import get_qiskit_compiler
from iqm.pulse.gates.prx import PRX_CustomWaveformsSX
from iqm.pulse.playlist.waveforms import Waveform

import iqm_tools


## Connect to IQM hardware

Everything from here on talks to a live server.


In [ ]:
# IQM_TOKEN must be set as an environment variable before running this notebook.
# Never hardcode a real token in a notebook cell, especially in a public repo.
#   export IQM_TOKEN="<your-api-token-here>"   # set this in your shell, before launching Jupyter
if 'IQM_TOKEN' not in os.environ:
    raise RuntimeError(
        "Set the IQM_TOKEN environment variable before running this notebook "
        "(e.g. `export IQM_TOKEN=<your-api-token-here>` in your shell before launching Jupyter). "
        "Never hardcode a token in a notebook cell, especially in a public repo."
    )

iqm_server_url = 'https://resonance.iqm.tech/emerald'   # <- change to your target machine

pulla   = Pulla(iqm_server_url)
backend = IQMProvider(iqm_server_url).get_backend()

print(f'Connected to {iqm_server_url}')


## Load the calibration

Compatible with the `*_calib.json` + `*_samples.npz` pair saved by `SX_calibration_exp.ipynb`.


In [ ]:
CAL_DIR = 'Results/Calibration'

calib_files = sorted(glob.glob(os.path.join(CAL_DIR, '*_calib.json')), key=os.path.getmtime)
print(f'{len(calib_files)} calibration file(s) found:\n')
for i, f in enumerate(calib_files):
    print(f'[{i}]  {os.path.basename(f)}')

idx = -1   # -1 = most recent
calib_path = calib_files[idx]

with open(calib_path) as f:
    calib = json.load(f)

samples_path = os.path.join(os.path.dirname(os.path.abspath(calib_path)), calib["samples_file"])
samples   = np.load(samples_path)
i_samples = samples["i_samples"]
q_samples = samples["q_samples"]

qbt       = calib["qubit"]
Texp      = calib["Texp_ns"]
Delta_ghz = calib.get("Delta_ghz", 0.0)

print(f"\nCalibration: {qbt}, Texp={Texp} ns, Delta={Delta_ghz*1e3:+.3f} MHz")
print(f"  amp_sx={calib['amp_sx']:.5f}  rz_before={calib.get('rz_before',0):.4f}  rz_after={calib.get('rz_after',0):.4f}")


In [ ]:
_I_SAMPLES = i_samples.copy()
_Q_SAMPLES = q_samples.copy()
_N         = len(_I_SAMPLES)

@dataclass(frozen=True)
class DakisSXWave_I(Waveform):
    def _sample(self, sample_coords: np.ndarray) -> np.ndarray:
        t_grid = np.linspace(0, 1, _N, endpoint=False)
        return np.interp(sample_coords + 0.5, t_grid, _I_SAMPLES,
                         left=_I_SAMPLES[0], right=_I_SAMPLES[-1])

@dataclass(frozen=True)
class DakisSXWave_Q(Waveform):
    def _sample(self, sample_coords: np.ndarray) -> np.ndarray:
        t_grid = np.linspace(0, 1, _N, endpoint=False)
        return np.interp(sample_coords + 0.5, t_grid, _Q_SAMPLES,
                         left=_Q_SAMPLES[0], right=_Q_SAMPLES[-1])

class DakisSXGate(PRX_CustomWaveformsSX, wave_i=DakisSXWave_I, wave_q=DakisSXWave_Q):
    """Leakage-suppressed sqrt(X) gate."""
    pass

compiler = get_qiskit_compiler(pulla, backend)
compiler.add_implementation("prx", "dakis_sx", DakisSXGate)

def get_dakis_sx_settings(circuits):
    s = compiler.get_settings(circuits=circuits)
    s.gate_definitions.prx.default_implementation = "dakis_sx"
    s.gates.prx.dakis_sx[qbt].set_from_dict({
        "duration":    Texp * 1e-9,
        "amplitude_i": calib["amplitude_i"],
        "amplitude_q": calib["amplitude_q"],
        "rz_before":   calib.get("rz_before", 0.0),
        "rz_after":    calib.get("rz_after",  0.0),
    })
    drive = s.controllers[qbt].drive
    drive.frequency = drive.frequency.value + Delta_ghz * 1e9
    return s

print("DakisSXGate ready.")


## Fix the native gate's current calibration

Reads the native implementation's *current* Settings and freezes them, so both arms of
the comparison stay fixed for the whole QPT run even if the live calibration set changes.


In [ ]:
_s_ref       = compiler.get_settings(circuits=[])
_native_impl = _s_ref.gate_definitions.prx.default_implementation.value
_prx_node    = _s_ref.gates.prx[_native_impl][qbt]

fixed_native_params = {
    'implementation': _native_impl,
    'duration':       _prx_node.duration.value,
    'amplitude_i':    _prx_node.amplitude_i.value,
    'amplitude_q':    _prx_node.amplitude_q.value,
    'rz_before':      _prx_node.rz_before.value,
    'rz_after':       _prx_node.rz_after.value,
    'full_width':     _prx_node.full_width.value,
    'center_offset':  _prx_node.center_offset.value,
}

def get_fixed_native_settings(circuits):
    s    = compiler.get_settings(circuits=circuits)
    impl = fixed_native_params['implementation']
    node = s.gates.prx[impl][qbt]
    node.duration      = fixed_native_params['duration']
    node.amplitude_i   = fixed_native_params['amplitude_i']
    node.amplitude_q   = fixed_native_params['amplitude_q']
    node.rz_before     = fixed_native_params['rz_before']
    node.rz_after      = fixed_native_params['rz_after']
    node.full_width    = fixed_native_params['full_width']
    node.center_offset = fixed_native_params['center_offset']
    return s

print(f"Native: {_native_impl}  ({fixed_native_params['duration']*1e9:.0f} ns)")


## QPT circuit structure

```
prep_gate | barrier | sx (TARGET) | barrier | meas_basis_gate | measure
```

- 4 input states: |0> (p0), |1> (p1), |+>=H|0> (p2), |+i>=S.H|0> (p3)
- 3 measurement bases: Z (m0), X=H.Z (m1), Y=H.Sdg.Z (m2) -> P(|1>) gives the Bloch vector of each output state
- + 2 REM circuits (|0> and |1> prepared, then measured) for readout calibration


In [ ]:
def make_qpt_circuits():
    records = []

    for p_idx in range(4):
        for m_idx in range(3):
            qc = QuantumCircuit(1, 1)

            # State preparation
            if   p_idx == 1: qc.x(0)             # |1>
            elif p_idx == 2: qc.h(0)             # |+>
            elif p_idx == 3: qc.h(0); qc.s(0)    # |+i>
            # p_idx == 0: nothing -> |0>

            qc.barrier(0)
            qc.sx(0)          # <- TARGET GATE
            qc.barrier(0)

            # Measurement basis rotation
            if   m_idx == 1: qc.h(0)             # X basis
            elif m_idx == 2: qc.sdg(0); qc.h(0)  # Y basis
            # m_idx == 0: nothing -> Z basis

            qc.measure(0, 0)
            qc.name = f'qpt_p{p_idx}_m{m_idx}'
            records.append({
                'name':     qc.name,
                'metadata': {'kind': 'qpt', 'prep_index': p_idx, 'meas_index': m_idx},
                'circuit':  qc,
            })

    # Readout error mitigation circuits
    for true_state in (0, 1):
        qc = QuantumCircuit(1, 1)
        if true_state == 1:
            qc.x(0)
        qc.measure(0, 0)
        qc.name = f'rem_{true_state}'
        records.append({
            'name':     qc.name,
            'metadata': {'kind': 'rem', 'true_state': true_state},
            'circuit':  qc,
        })

    return records

records = make_qpt_circuits()

print(f"{len(records)} circuits total ({len(records)-2} QPT + 2 REM)\n")
for r in records:
    print(r['name'], ':', r['metadata'])


## Run QPT

In [ ]:
def run_qpt_batched(records, gate, n_shots):
    """
    Compiles + submits ALL circuits (REM + QPT) in ONE playlist -- one job submission
    instead of one per circuit.
    gate: 'dakis' | 'native_fixed' | 'native'
    """
    assert gate in ('dakis', 'native_fixed', 'native')
    print(f"Running batched QPT  [{gate}]  {n_shots} shots/circuit ...")

    qcs = [r['circuit'] for r in records]

    if gate == 'dakis':
        s = get_dakis_sx_settings(qcs)
    elif gate == 'native_fixed':
        s = get_fixed_native_settings(qcs)
    else:
        s = compiler.get_settings(circuits=qcs)
    s.set_shots(n_shots)

    jd, ctx = compiler.compile(circuits=qcs, components=[qbt], settings=s)
    job = pulla.submit_playlist(jd, context=ctx)
    job.wait_for_completion()

    if job.status != 'completed':
        raise RuntimeError(f'Batched QPT job failed: {job.status}')

    res = job.result(compiler)

    results = []
    for i, r in enumerate(records):
        prob_raw  = float(res.dataset['counter.result'].isel(circuit_index=i).values.flat[-1])
        prob_corr = float(res.dataset[f'{qbt}__c_1_0_0_excited_state_probability'].isel(circuit_index=i).values.flat[0])
        print(f'  {r["name"]:20s}  raw={prob_raw:.4f}  corr={prob_corr:.4f}')
        results.append({**r, 'prob_raw': prob_raw, 'prob_corr': prob_corr})

    return results


In [ ]:
n_shots = 1000

results_dakis = run_qpt_batched(records, gate='dakis', n_shots=n_shots)


In [ ]:
results_native = run_qpt_batched(records, gate='native_fixed', n_shots=n_shots)


## Process reconstruction

Each circuit measures P(|1>) for one (prep, basis) pair. From 3 measurements per input
state we get the full output Bloch vector, from which the Choi matrix is assembled.

Verified against real saved data before use in this notebook -- see the intro cell.


In [ ]:
I2 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)    # Pauli X
sy = np.array([[0, -1j], [1j, 0]], dtype=complex) # Pauli Y
sz = np.array([[1, 0], [0, -1]], dtype=complex)   # Pauli Z

def rem_correct(prob_raw, p10, p01):
    """Simple 2-point linear readout error correction."""
    denom = 1 - p01 - p10
    if abs(denom) < 1e-6:
        return prob_raw
    return np.clip((prob_raw - p10) / denom, 0, 1)

def reconstruct_choi(results, use_corrected=True):
    """
    Reconstruct the Choi matrix from QPT measurement results.
    Returns: (C, p_matrix, output_states)
      C             -- 4x4 Choi matrix (complex)
      p_matrix      -- 4x3 array of P(|1>) values
      output_states -- list of 4 output density matrices
    """
    prob_key = 'prob_corr' if use_corrected else 'prob_raw'

    rem0 = next(r for r in results if r['metadata'].get('true_state') == 0)
    rem1 = next(r for r in results if r['metadata'].get('true_state') == 1)
    p10  = rem0[prob_key]       # P(meas=1 | prep=|0>)  -- flip-up error
    p01  = 1 - rem1[prob_key]   # P(meas=0 | prep=|1>)  -- flip-down error
    print(f'  REM: p10={p10:.4f}  p01={p01:.4f}')

    p = np.full((4, 3), np.nan)
    for r in results:
        meta = r['metadata']
        if meta.get('kind') != 'qpt':
            continue
        j, k = meta['prep_index'], meta['meas_index']
        p[j, k] = rem_correct(r[prob_key], p10, p01)

    sigma = []
    for j in range(4):
        x_exp = 1 - 2 * p[j, 1]   # <X>
        y_exp = 1 - 2 * p[j, 2]   # <Y>
        z_exp = 1 - 2 * p[j, 0]   # <Z>
        sigma.append((I2 + x_exp*sx + y_exp*sy + z_exp*sz) / 2)

    A   = 2*sigma[2] - sigma[0] - sigma[1]
    B   = -1j * (2*sigma[3] - sigma[0] - sigma[1])
    L01 = (A - B) / 2   # Lambda(|0><1|)
    L10 = (A + B) / 2   # Lambda(|1><0|)

    C = np.block([[sigma[0], L01],
                  [L10,      sigma[1]]])  # Tr(C)=d=2, qiskit Choi convention

    return C, p, sigma

print("reconstruct_choi() ready")


In [ ]:
use_corrected = False   # set True to use readout-corrected probabilities

ideal_choi = Choi(Operator(SXGate()))

print("-- DakisSX --")
C_dakis, p_dakis, sigma_dakis = reconstruct_choi(results_dakis, use_corrected)
F_dakis = process_fidelity(Choi(C_dakis), ideal_choi).real
print(f"  Process fidelity: {F_dakis:.4f}")

print("-- Native (fixed) --")
C_native, p_native, sigma_native = reconstruct_choi(results_native, use_corrected)
F_native = process_fidelity(Choi(C_native), ideal_choi).real
print(f"  Process fidelity: {F_native:.4f}")


In [ ]:
prep_labels = ['|0>', '|1>', '|+>', '|+i>']
meas_labels = ['Z', 'X', 'Y']

datasets = [('DakisSX', p_dakis), ('Native (fixed)', p_native)]

fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 4))

for ax, (label, p_data) in zip(axes, datasets):
    im = ax.imshow(p_data, vmin=0, vmax=1, cmap='RdBu_r', aspect='auto')
    ax.set_xticks(range(3)); ax.set_xticklabels(meas_labels)
    ax.set_yticks(range(4)); ax.set_yticklabels(prep_labels)
    ax.set_xlabel('Measurement basis'); ax.set_ylabel('Input state')
    ax.set_title(f'P(|1>) -- {label}')
    plt.colorbar(im, ax=ax)
    for j in range(4):
        for k in range(3):
            ax.text(k, j, f'{p_data[j,k]:.3f}', ha='center', va='center', fontsize=8)

plt.suptitle(f'QPT measurement table  |  {qbt}', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
ideal_C = np.array(ideal_choi.data)

choi_datasets = [('Ideal SX', ideal_C), ('DakisSX', C_dakis), ('Native (fixed)', C_native)]

n_cols = len(choi_datasets)
fig, axes = plt.subplots(2, n_cols, figsize=(5*n_cols, 8))

for col, (title, C) in enumerate(choi_datasets):
    for row, (part, label) in enumerate([(np.real, 'Re'), (np.imag, 'Im')]):
        ax = axes[row, col]
        im = ax.imshow(part(C), vmin=-0.5, vmax=0.5, cmap='RdBu')
        ax.set_title(f'{title}  ({label})')
        ax.set_xticks(range(4)); ax.set_yticks(range(4))
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(f'Choi matrices  |  {qbt}  |  F_dakis={F_dakis:.4f}  F_native={F_native:.4f}', fontsize=10)
plt.tight_layout()
plt.show()


## Save the result

In [ ]:
expr_params  = iqm_tools.get_qubit_params(iqm_server_url)
qbt_row      = expr_params[expr_params['qubit'] == qbt].iloc[0]
calib_set_id = expr_params.attrs.get('calibration_set_id', 'unknown')
machine      = iqm_server_url.rstrip('/').split('/')[-1]

def _serialise_results(results):
    return [
        {'name': r['name'], 'metadata': r['metadata'],
         'prob_raw': r['prob_raw'], 'prob_corr': r['prob_corr']}
        for r in results
    ]

record = {
    'timestamp':          datetime.now().isoformat(),
    'machine':            machine,
    'calibration_set_id': calib_set_id,
    'qubit':              qbt,
    'T1_us':              float(qbt_row['T1_us']),
    'T2_ramsey_us':       float(qbt_row['T2_ramsey_us']),
    'n_shots':            n_shots,
    'dakis_calibration':  calib,
    'native_gate':        fixed_native_params,
    'process_fidelity': {
        'dakis':  float(F_dakis),
        'native': float(F_native),
    },
    'choi_matrix': {
        'dakis_real':  C_dakis.real.tolist(),
        'dakis_imag':  C_dakis.imag.tolist(),
        'native_real': C_native.real.tolist(),
        'native_imag': C_native.imag.tolist(),
        'ideal_real':  ideal_C.real.tolist(),
        'ideal_imag':  ideal_C.imag.tolist(),
    },
    'results_dakis':  _serialise_results(results_dakis),
    'results_native': _serialise_results(results_native),
}

ts        = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir  = 'Results/QPT'
save_path = f'{save_dir}/qpt_sx_{qbt}_{machine}_{ts}.json'
os.makedirs(save_dir, exist_ok=True)
with open(save_path, 'w') as f:
    json.dump(record, f, indent=2)

print(f'Saved: {save_path}')
print(f'  F_dakis={F_dakis:.4f}   F_native={F_native:.4f}')
